# Постобработка OCR паспорта РФ

Проверка 10 полей, MRZ и срока действия на 01.10.2026. Ноутбук работает без сети и не печатает персональные данные.

In [1]:
import json
from collections import Counter
import postprocess as pp

with open("data/train_predictions.jsonl", encoding="utf-8") as f:
    train = [json.loads(line) for line in f]
with open("data/train_labels.jsonl", encoding="utf-8") as f:
    truth = {d["doc_id"]: d["fields"] for d in map(json.loads, f)}
with open("data/test_predictions.jsonl", encoding="utf-8") as f:
    test = [json.loads(line) for line in f]
print("train:", len(train), "test:", len(test))

train: 3000 test: 2000


## Критерий

Принятый документ считается верным, когда совпали все 10 полей и паспорт действителен. `evaluate.py` проверяет только поля; срок действия проверяет `postprocess.py`.

In [2]:
def count_results(docs, answers):
    good = bad = 0
    for doc, answer in zip(docs, answers):
        if answer["decision"] != "accept":
            continue
        fields = truth[doc["doc_id"]]
        exact = all(pp.norm(answer["fields"].get(k)) == pp.norm(fields[k]) for k in pp.FIELDS)
        valid = pp.valid_on_date(fields)
        good += bool(exact and valid)
        bad += not (exact and valid)
    return good, bad

def assess(name, kwargs):
    answers = [pp.process(d, **kwargs) for d in train]
    print(f"{name:19} первые 2000: {count_results(train[:2000], answers[:2000])}, последние 1000: {count_results(train[2000:], answers[2000:])}")
    return answers

In [3]:
raw = assess("Визуальная зона", dict(thresholds={f: 0 for f in pp.FIELDS}, require_mrz=False, include_mobile_v2=True))
mrz = assess("MRZ + порог 0.90", dict(thresholds={f: 0.90 for f in pp.FIELDS}))
strict = assess("Итоговые правила", {})

Визуальная зона     первые 2000: (1062, 552), последние 1000: (534, 266)


MRZ + порог 0.90    первые 2000: (781, 13), последние 1000: (401, 7)


Итоговые правила    первые 2000: (90, 0), последние 1000: (49, 0)


## Где ошибается сырой OCR

Считаем расхождения полей для всех train документов. Частые ошибки в длинных полях и при выпадении строки.

In [4]:
field_errors = Counter()
missing = Counter()
for doc in train:
    for field in pp.FIELDS:
        value, _ = pp.read_field(doc["viz"].get(field))
        if value is None:
            missing[field] += 1
        if pp.norm(value) != pp.norm(truth[doc["doc_id"]][field]):
            field_errors[field] += 1
for field in pp.FIELDS:
    print(f"{field:28} ошибочных/неполных: {field_errors[field]:4}, не прочитано: {missing[field]:3}")

Last_name_ru                 ошибочных/неполных:  136, не прочитано:  23
First_name_ru                ошибочных/неполных:   96, не прочитано:  19
Middle_name_ru               ошибочных/неполных:  140, не прочитано: 141
Sex_ru                       ошибочных/неполных:   47, не прочитано:  27
Birth_date                   ошибочных/неполных:  124, не прочитано:  25
Birth_place_ru               ошибочных/неполных:  519, не прочитано:  57
Issue_date                   ошибочных/неполных:  113, не прочитано:  14
Issue_organisation_code      ошибочных/неполных:  108, не прочитано:  27
Licence_number               ошибочных/неполных:  156, не прочитано:  28
Issue_organization_ru        ошибочных/неполных:  602, не прочитано:  62


## Прогноз для test

Используется ровно тот же код, что создает `answers.jsonl`. Ноль ошибок на train не гарантирует ноль на test. Пороги подбирались с просмотром train, поэтому train-оценка оптимистична.

In [5]:
test_answers = [pp.process(d) for d in test]
print(Counter(a["decision"] for a in test_answers))
print("Прогноз: принято", sum(a["decision"] == "accept" for a in test_answers), "документа; точечная оценка ошибок 0, гарантий нет.")

Counter({'reject': 1917, 'accept': 83})
Прогноз: принято 83 документа; точечная оценка ошибок 0, гарантий нет.
